# Staging
## Transformation & Nettoyage des données

**Objectif :** Transformer les données brutes en vues propres et typées.

**Principe :** Matérialisées en VIEW, pas de stockage redondant.
Les données sont toujours fraîches car recalculées à chaque requête.

**Transformations appliquées :**
- Typage explicite des colonnes (CAST)
- Suppression des doublons (QUALIFY ROW_NUMBER)
- Gestion des valeurs nulles (COALESCE)
- Conversion des dates françaises (COALESCE + to_date)
- Filtrage des joueurs non sélectionnés (COMMENT IS NULL)
- Imputation par moyenne segment (AVG OVER PARTITION BY)

| Vue | Source | Lignes | Clé |
|---|---|---|---|
| staging.team_players_stats | raw.team_players_stats | 4 280 | player_id + game_id |
| staging.team_training_sessions | raw.team_training_sessions | 9464 | session_id |
| staging.team_players_personal_info | raw.team_players_personal_info | 60 | player_id |
| staging.team_games_dataset | raw.team_games_dataset | 389 | game_id |

### staging.team_players_stats

**Transformations clés :**
- Suppression joueurs non sélectionnés → `WHERE COMMENT IS NULL`
- Conversion minutes `MM:SS` → décimales
- Déduplication → `QUALIFY ROW_NUMBER() OVER (PARTITION BY player_id, game_id)`
- Colonne `TO` protégée avec backticks → mot réservé Databricks

In [0]:
-- ============================================
-- Staging : team_players_stats
-- Filtrage : joueurs non sélectionnés exclus
-- Déduplication : QUALIFY ROW_NUMBER
-- Particularité : TO est un mot réservé → backticks
-- ============================================
CREATE OR REPLACE VIEW sport_metrics.staging.team_players_stats AS
WITH donnees_source AS (
    SELECT * FROM sport_metrics.raw.team_players_stats
),
nettoyage AS (
    SELECT
        CAST(GAME_ID AS INT)        AS game_id,
        CAST(PLAYER_ID AS INT)      AS player_id,
        CASE
            WHEN TRIM(START_POSITION) = ''
              OR START_POSITION IS NULL THEN 'Bench'
            ELSE TRIM(START_POSITION)
        END AS start_position,
        ROUND(CAST(MIN AS FLOAT) * 60, 2) AS minutes_played,
        CAST(PTS   AS INT)          AS points,
        CAST(FGM   AS INT)          AS field_goal_made,
        CAST(FGA   AS INT)          AS field_goal_attempt,
        CAST(FG_PCT  AS FLOAT)      AS fg_pct,
        CAST(FG3M  AS INT)          AS field_goal_3pts_made,
        CAST(FG3A  AS INT)          AS field_goal_3pts_attempt,
        CAST(FG3_PCT AS FLOAT)      AS fg3_pct,
        CAST(FTM   AS INT)          AS free_throws_made,
        CAST(FTA   AS INT)          AS free_throws_attempt,
        CAST(FT_PCT  AS FLOAT)      AS ft_pct,
        CAST(OREB  AS INT)          AS offensive_rebounds,
        CAST(DREB  AS INT)          AS defensive_rebounds,
        CAST(REB   AS INT)          AS total_rebounds,
        CAST(AST   AS INT)          AS assists,
        CAST(STL   AS INT)          AS steals,
        CAST(BLK   AS INT)          AS blocks,
        CAST(`TO`  AS INT)          AS turnover,
        CAST(PF    AS INT)          AS player_fault,
        CAST(PLUS_MINUS AS FLOAT)   AS plus_minus
    FROM donnees_source
    WHERE COMMENT IS NULL
),
doublons AS (
    SELECT * FROM nettoyage
    WHERE player_id IS NOT NULL
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY player_id, game_id
        ORDER BY player_id DESC
    ) = 1
)
SELECT * FROM doublons;

### staging.team_players_personal_info

**Transformations clés :**
- Imputation de la position par taille → `CASE WHEN height_cm > 200`
- Imputation taille/poids par âge → `AVG OVER (PARTITION BY AGE)`
- Déduplication → `QUALIFY ROW_NUMBER() OVER (PARTITION BY player_id)`

In [0]:
-- ============================================
-- Staging : team_players_personal_info
-- Imputation : position déduite de la taille si manquante
-- Imputation : taille/poids par moyenne d'âge
-- ============================================
CREATE OR REPLACE VIEW sport_metrics.staging.team_players_personal_info AS
WITH donnees_source AS (
    SELECT * FROM sport_metrics.raw.team_players_personal_info
),
nettoyage AS (
    SELECT
        PLAYER_ID                               AS player_id,
        PLAYER_NAME                             AS player_name,
        FIRST_NAME                              AS first_name,
        LAST_NAME                               AS last_name,
        DATE(CAST(BIRTHDATE AS TIMESTAMP))      AS birthdate,
        AGE                                     AS age,
        COALESCE(HEIGHT_CM, AVG(HEIGHT_CM) OVER (PARTITION BY AGE)) AS height_cm,
        COALESCE(WEIGHT_KG, AVG(WEIGHT_KG) OVER (PARTITION BY AGE)) AS weight_kg,
        COALESCE(
            NULLIF(POSITION, ''),
            CASE
                WHEN COALESCE(HEIGHT_CM, AVG(HEIGHT_CM) OVER (PARTITION BY AGE)) > 200 THEN 'Forward'
                WHEN COALESCE(HEIGHT_CM, AVG(HEIGHT_CM) OVER (PARTITION BY AGE)) > 0   THEN 'Guard'
                ELSE 'Inconnu'
            END
        )                                       AS position,
        SCHOOL,
        COUNTRY,
        CAST(SEASON_EXP AS INT)                 AS season_exp
    FROM donnees_source
    WHERE PLAYER_ID IS NOT NULL
      AND BIRTHDATE IS NOT NULL
      AND AGE IS NOT NULL
),
doublons AS (
    SELECT * FROM nettoyage
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY player_id
        ORDER BY player_id DESC
    ) = 1
)
SELECT * FROM doublons;

### staging.team_games_dataset

**Transformations clés :**
- Dates en français → `COALESCE + to_date + replace` (parse_date_fr)
- Jointure avec raw.team_boxscores pour récupérer PLUS_MINUS
- Extraction adversaire depuis Matchup → `SPLIT_PART`

In [0]:
-- ============================================
-- Staging : team_games_dataset
-- Particularité : dates en français → parse manuelle
-- Jointure : team_boxscores pour PLUS_MINUS
-- ============================================
CREATE OR REPLACE VIEW sport_metrics.staging.team_games_dataset AS
WITH donnees AS (
  SELECT
    GAME_ID,
    PLUS_MINUS
  FROM sport_metrics.raw.team_boxscores
),

game_data AS (
  SELECT
    CAST(g.GAME_ID AS INT) AS game_id,

    COALESCE(

      -- Dates texte
      try_to_date(
        replace(replace(replace(trim(g.GAME_DATE),
          'oct.','Oct'),'nov.','Nov'),'déc.','Dec'),
        'MMM dd, yyyy'
      ),

      try_to_date(
        replace(replace(replace(trim(g.GAME_DATE),
          'janv.','Jan'),'févr.','Feb'),'avr.','Apr'),
        'MMM dd, yyyy'
      ),

      try_to_date(
        replace(replace(replace(trim(g.GAME_DATE),
          'mai','May'),'juin','Jun'),'juil.','Jul'),
        'MMM dd, yyyy'
      ),

      try_to_date(
        replace(replace(replace(trim(g.GAME_DATE),
          'août','Aug'),'sept.','Sep'),'mars','Mar'),
        'MMM dd, yyyy'
      ),

      -- Dates Excel
      date_add(
        DATE '1899-12-30',
        try_cast(g.GAME_DATE AS INT)
      )

    ) AS game_date,

    g.Matchup,
    g.WL AS win_loss,
    g.W AS win,
    g.L AS loss,
    g.W_PCT AS win_pct,
    g.MIN AS total_minutes,
    CAST(g.PTS AS INT) AS total_points,
    g.FGM AS total_field_goal_made,
    g.FGA AS total_field_goal_attempt,
    g.FG_PCT,
    g.FG3M AS total_field_goal_3pts_made,
    g.FG3A AS total_field_goal_3pts_attempt,
    g.FG3_PCT,
    g.FTM AS total_free_throws_made,
    g.FTA AS total_free_throws_attempt,
    g.FT_PCT,
    g.OREB AS total_offensive_rebounds,
    g.DREB AS total_defensive_rebounds,
    g.REB AS total_rebounds,
    g.AST AS total_assists,
    g.STL AS total_steals,
    g.BLK AS total_blocks,
    g.TOV AS total_turnover,
    g.PF AS total_player_fault,
    CAST(d.PLUS_MINUS AS INT) AS ecart

  FROM sport_metrics.raw.team_games_dataset g
  JOIN donnees d
    ON CAST(g.GAME_ID AS INT) = CAST(d.GAME_ID AS INT)
)

SELECT *
FROM game_data;

### staging.team_training_sessions

**Transformations clés :**
- Jointure avec staging.team_players_personal_info pour l'âge et la position
- Imputation par moyenne → `AVG OVER (PARTITION BY age, position)`
- Calcul de la saison NBA → `CASE WHEN session_date <= DATE '2020-07-31'`
- Filtres qualité : sessions incomplètes exclues

In [0]:
-- ============================================
-- Staging : team_training_sessions
-- Imputation : métriques physiologiques par âge/position
-- Calcul saison NBA : CASE WHEN sur session_date
-- Filtres : sessions sans player_id, age, session_id exclus
-- ============================================
CREATE OR REPLACE VIEW sport_metrics.staging.team_training_sessions AS
WITH donnees_source AS (
    SELECT * FROM sport_metrics.raw.team_training_sessions
),
pi AS (
    SELECT * FROM sport_metrics.staging.team_players_personal_info
),
nettoyage AS (
    SELECT
        d.SESSION_ID                                AS session_id,
        CAST(d.PLAYER_ID AS INT)                    AS player_id,
        CAST(d.NEXT_MATCH_ID AS INT)                AS next_match_id,
        DATE(CAST(d.SESSION_DATE AS TIMESTAMP))     AS session_date,
        CAST(d.DURATION_MIN AS FLOAT)               AS duration_min,
        COALESCE(CAST(d.HEART_RATE AS FLOAT),
            AVG(CAST(d.HEART_RATE AS FLOAT)) OVER (
                PARTITION BY pi.age, pi.position))  AS heart_rate,
        COALESCE(CAST(d.STRENGTH_SCORE AS FLOAT),
            AVG(CAST(d.STRENGTH_SCORE AS FLOAT)) OVER (
                PARTITION BY pi.age, pi.position))  AS strength_score,
        COALESCE(CAST(d.`Shooting_Accuracy_%` AS FLOAT),
            AVG(CAST(d.`Shooting_Accuracy_%` AS FLOAT)) OVER (
                PARTITION BY pi.age, pi.position))  AS shooting_accuracy_pct,
        CAST(d.`Passing_Accuracy_%` AS FLOAT)       AS passing_accuracy_pct,
        CAST(d.FOCUS_LEVEL AS FLOAT)                AS focus_level,
        CAST(d.WEEKLY_TRAINING_HOURS AS FLOAT)      AS weekly_training_hours,
        COALESCE(CAST(d.LOAD_INTENSITY_SCORE AS FLOAT),
            AVG(CAST(d.LOAD_INTENSITY_SCORE AS FLOAT)) OVER (
                PARTITION BY pi.age, pi.position))  AS load_intensity_score,
        COALESCE(d.FATIGUE_LEVEL, 'Low')            AS fatigue_level,
        COALESCE(CAST(d.INJURY_RISK AS INT), 0)     AS injury_risk,
        COALESCE(d.INJURY_RISK_LEVEL, 'Low')        AS injury_risk_level,
        CAST(d.RECOVERY_TIME_HOURS AS FLOAT)        AS recovery_time_hours,
        COALESCE(CAST(d.PERFORMANCE_SCORE AS FLOAT),
            AVG(CAST(d.PERFORMANCE_SCORE AS FLOAT)) OVER (
                PARTITION BY pi.age, pi.position))  AS performance_score,
        CAST(d.DAYS_BEFORE_MATCH AS INT)            AS days_before_match
    FROM donnees_source d
    LEFT JOIN pi ON pi.player_id = CAST(d.PLAYER_ID AS INT)
    WHERE d.PLAYER_ID IS NOT NULL
      AND pi.age IS NOT NULL
      AND d.SESSION_ID IS NOT NULL
      AND d.DAYS_BEFORE_MATCH IS NOT NULL
      AND d.RECOVERY_TIME_HOURS IS NOT NULL
      AND d.DURATION_MIN IS NOT NULL
)
SELECT
    CASE
        WHEN session_date <= DATE '2020-07-31' THEN '2019-2020'
        WHEN session_date <= DATE '2021-07-31' THEN '2020-2021'
        WHEN session_date <= DATE '2022-07-31' THEN '2021-2022'
        WHEN session_date <= DATE '2023-07-31' THEN '2022-2023'
        ELSE '2023-2024'
    END AS season,
    n.*
FROM nettoyage n;

In [0]:
-- ============================================
-- Vérification : nombre de lignes par vue
-- ============================================
SELECT 'staging.team_players_stats'         AS vue, COUNT(*) AS nb_lignes FROM sport_metrics.staging.team_players_stats
UNION ALL
SELECT 'staging.team_players_personal_info' AS vue, COUNT(*) AS nb_lignes FROM sport_metrics.staging.team_players_personal_info
UNION ALL
SELECT 'staging.team_games_dataset'         AS vue, COUNT(*) AS nb_lignes FROM sport_metrics.staging.team_games_dataset
UNION ALL
SELECT 'staging.team_training_sessions'     AS vue, COUNT(*) AS nb_lignes FROM sport_metrics.staging.team_training_sessions
ORDER BY nb_lignes DESC;